In [80]:
from IPython.display import HTML, display

display(HTML("""
<div style="background: linear-gradient(135deg, #1a1a2e 0%, #16213e 60%, #0f3460 100%);
            border-radius: 12px; padding: 40px 40px 30px 40px; margin: 10px 0 30px 0;
            font-family: 'Helvetica Neue', Arial, sans-serif;">
  <div style="color:#a0aec0; font-size:0.8em; letter-spacing:2px; text-transform:uppercase; margin-bottom:12px;">
    LLM Lab — Section 01
  </div>
  <h1 style="color:white; font-size:2.2em; margin:0 0 10px 0; font-weight:700; line-height:1.2; border:none;">
    MLX Inference<br>
    <span style="color:#63b3ed;">3-Model Comparison on Your Mac</span>
  </h1>
  <p style="color:#cbd5e0; font-size:1em; margin:16px 0 24px 0; max-width:600px; line-height:1.6;">
    Apple Silicon's unified memory changes everything. Run 122B, 35B, and 2B
    models side-by-side to see the quality/speed tradeoff viscerally.
  </p>
  <div style="display:flex; gap:16px; flex-wrap:wrap;">
    <span style="background:rgba(255,255,255,0.1); color:#e2e8f0; padding:6px 14px; border-radius:20px; font-size:0.85em;">🍎 Apple Silicon</span>
    <span style="background:rgba(255,255,255,0.1); color:#e2e8f0; padding:6px 14px; border-radius:20px; font-size:0.85em;">🧠 MLX</span>
    <span style="background:rgba(255,255,255,0.1); color:#e2e8f0; padding:6px 14px; border-radius:20px; font-size:0.85em;">🔌 OpenAI API</span>
    <span style="background:rgba(255,255,255,0.1); color:#e2e8f0; padding:6px 14px; border-radius:20px; font-size:0.85em;">🏆 3 Models</span>
    <span style="background:rgba(255,255,255,0.1); color:#e2e8f0; padding:6px 14px; border-radius:20px; font-size:0.85em;">⏱ ~20 min</span>
  </div>
</div>
"""))

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #4f8ef7; background:#f0f4ff; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">💻 Step 1: Hardware Check</h2>
</div>

First — what are we working with?

In [81]:
!python3 ../../scripts/setup_check.py


┌────────────────────────────────────────────────────────────┐
│                    Hardware Setup Check                    │
└────────────────────────────────────────────────────────────┘

  ▸ Operating System
  ────────────────────────────────────────────────────────
    OS                           Darwin 25.3.0
    Architecture                 arm64
    Python                       3.14.3

  ▸ Chip / Accelerator
  ────────────────────────────────────────────────────────
    Chip                         Apple M4 Max

  ▸ Memory (RAM)
  ────────────────────────────────────────────────────────
    Total RAM                    128.0 GiB
    Available RAM                105.0 GiB

  ▸ MLX Framework
  ────────────────────────────────────────────────────────
    ✓  mlx.core importable                       version=unknown  device=Device(gpu, 0)

  ▸ Model Recommendations
  ────────────────────────────────────────────────────────
    Detected tier: 128 GB+

    Model                      

In [82]:
# Auto-detect running MLX servers and warm up
# Works with 1, 2, or 3 models — adapts to whatever's running on your machine.
# Start servers with: mlx_lm.server --model <model_id> --port <port>
import socket, threading, time
from IPython.display import display, HTML, update_display
from openai import OpenAI

# Registry of known models — add/remove as needed
ALL_MODELS = [
    {"label": "122B", "model": "arthurcollet/Qwen3.5-122B-A10B-mlx-nvfp4",          "port": 8800, "color": "#2563eb"},
    {"label": "35B",  "model": "RepublicOfKorokke/Qwen3.5-35B-A3B-mlx-lm-nvfp4",    "port": 8801, "color": "#16a34a"},
    {"label": "2B",   "model": "RepublicOfKorokke/Qwen3.5-2B-mlx-lm-nvfp4",         "port": 8802, "color": "#f59e0b"},
]

def _port_open(port):
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.settimeout(0.5)
    ok = s.connect_ex(("localhost", port)) == 0
    s.close()
    return ok

# Discover which servers are actually running
print("Scanning ports...", end=" ", flush=True)
MODELS = [m for m in ALL_MODELS if _port_open(m["port"])]
skipped = [m for m in ALL_MODELS if not _port_open(m["port"])]

if not MODELS:
    raise RuntimeError(
        "No MLX servers found! Start at least one:\n"
        + "\n".join(f'  mlx_lm.server --model {m["model"]} --port {m["port"]}' for m in ALL_MODELS)
    )

for m in skipped:
    label = m["label"]
    port = m["port"]
    print(f"\n  {label} (port {port}): not running — skipping", end="")
print(f"\nFound {len(MODELS)} model{'s' if len(MODELS) != 1 else ''}: {', '.join(m['label'] for m in MODELS)}")

# State & rendering
warmup_state = {}
for m in MODELS:
    warmup_state[m["label"]] = {"status": "warming", "elapsed": 0.0, "start": time.time()}

_CSS = "<style>@keyframes spin{to{transform:rotate(360deg)}}.sp{display:inline-block;width:14px;height:14px;border:2px solid #ccc;border-top-color:#555;border-radius:50%;animation:spin .8s linear infinite;vertical-align:middle}</style>"

def _render():
    rows = []
    for m in MODELS:
        label = m["label"]
        color = m["color"]
        port = m["port"]
        st = warmup_state[label]
        status = st["status"]
        elapsed = time.time() - st["start"]
        if status == "warming":
            cell = '<span class="sp"></span> warming up...'
            el = f"{elapsed:.0f}s"
        elif status == "done":
            cell = '<span style="color:#16a34a">&#x2713; ready</span>'
            el = f"{st['elapsed']:.1f}s"
        else:
            cell = '<span style="color:#dc2626">&#x2717; failed</span>'
            el = f"{elapsed:.0f}s"
        rows.append(
            f'<tr>'
            f'<td style="padding:6px 12px;font-weight:bold;color:{color}">{label}</td>'
            f'<td style="padding:6px 12px">{port}</td>'
            f'<td style="padding:6px 12px">{cell}</td>'
            f'<td style="padding:6px 12px;color:#888">{el}</td>'
            f'</tr>'
        )
    header = (
        '<thead><tr style="background:#1e3a5f">'
        '<th style="padding:8px 12px;color:white;text-align:left">Model</th>'
        '<th style="padding:8px 12px;color:white;text-align:left">Port</th>'
        '<th style="padding:8px 12px;color:white;text-align:left">Status</th>'
        '<th style="padding:8px 12px;color:white;text-align:left">Time</th>'
        '</tr></thead>'
    )
    body = "\n".join(rows)
    return (
        _CSS
        + '<table style="border-collapse:collapse;font-family:monospace;font-size:14px;'
        + 'background:#f9fafb;border:1px solid #d1d5db;border-radius:8px;overflow:hidden;min-width:500px">'
        + header + '<tbody>' + body + '</tbody></table>'
    )

# Create clients for running servers only
clients = {}
for m in MODELS:
    clients[m["label"]] = OpenAI(base_url=f"http://localhost:{m['port']}/v1", api_key="unused")

def _warmup(m):
    label = m["label"]
    warmup_state[label]["start"] = time.time()
    try:
        resp = clients[label].chat.completions.create(
            model=m["model"],
            messages=[{"role": "user", "content": "hi"}],
            max_tokens=3,
            stream=True,
        )
        for chunk in resp:
            pass
        warmup_state[label]["elapsed"] = time.time() - warmup_state[label]["start"]
        warmup_state[label]["status"] = "done"
    except Exception:
        warmup_state[label]["status"] = "failed"

dh = display(HTML(_render()), display_id=True)

threads = [threading.Thread(target=_warmup, args=(m,), daemon=True) for m in MODELS]
for t in threads:
    t.start()

while any(t.is_alive() for t in threads):
    time.sleep(0.5)
    update_display(HTML(_render()), display_id=dh.display_id)

for t in threads:
    t.join()

update_display(HTML(_render()), display_id=dh.display_id)

ready = [m["label"] for m in MODELS if warmup_state[m["label"]]["status"] == "done"]
failed = [m["label"] for m in MODELS if warmup_state[m["label"]]["status"] == "failed"]
if failed:
    print(f"Failed: {', '.join(failed)}")
if ready and not failed:
    print(f"All {len(ready)} models ready!")
elif ready:
    print(f"Ready: {', '.join(ready)}")


Scanning ports... 
  35B (port 8801): not running — skipping
Found 2 models: 122B, 2B


Model,Port,Status,Time
122B,8800,✓ ready,20.0s
2B,8802,✓ ready,0.2s


All 2 models ready!


In [83]:
# Helper functions for 3-model comparisons
import re
import time
import html
import markdown
import threading
import concurrent.futures
from IPython.display import HTML, display

def strip_think(text):
    """Remove <think>...</think> blocks from Qwen3.5 reasoning output."""
    cleaned = re.sub(r'<think>.*?</think>\s*', '', text, flags=re.DOTALL)
    if '<think>' in cleaned:
        cleaned = cleaned[:cleaned.index('<think>')]
    return cleaned.strip()

def _md(text):
    """Convert markdown text to HTML. Falls back to escaped text on error."""
    try:
        return markdown.markdown(text, extensions=["fenced_code", "tables"])
    except Exception:
        return html.escape(text)

def _render_cards(state, models_order):
    """Render 3-column HTML cards from current state."""
    cards = ""
    for m in models_order:
        s = state[m["label"]]
        text = strip_think(s["text"])
        if not text and not s["done"]:
            rendered = "<em>waiting...</em>"
        elif not text and s["done"]:
            rendered = "<em>(empty response)</em>"
        else:
            rendered = _md(text)
        if s["done"]:
            ttft_ms = int(s["ttft"] * 1000) if s["ttft"] is not None else None
            ttft_str = f"TTFT {ttft_ms}ms · " if ttft_ms is not None else ""
            status = f"{ttft_str}{s['tps']:.1f} tok/s"
        else:
            ttft = s["ttft"]
            elapsed_str = f"{s['elapsed']:.1f}s" if s["elapsed"] > 0 else ""
            if ttft is not None:
                ttft_ms = int(ttft * 1000)
                status = f"streaming... TTFT: {ttft_ms}ms · {s['tokens']} tok {elapsed_str}"
            else:
                status = f"streaming... {s['tokens']} tok {elapsed_str}"
        cards += f"""
        <div style="flex:1; min-width:250px; background:#f9fafb; border:1px solid #d1d5db;
                    border-left:4px solid {s['color']}; border-radius:0 8px 8px 0; padding:12px 18px;">
          <div style="color:{s['color']}; font-weight:bold; font-size:0.85em; margin-bottom:6px;">
            {m['label']} · {status}
          </div>
          <div style="color:#1f2937; line-height:1.6; word-wrap:break-word; font-size:0.9em;">
            {rendered}
          </div>
          <div style="color:#9ca3af; font-size:0.75em; margin-top:8px;">
            {s['tokens']} tokens in {s['elapsed']:.1f}s
          </div>
        </div>"""
    return f'<div style="display:flex; gap:16px; flex-wrap:wrap; margin:10px 0;">{cards}</div>'

def compare_models(prompt, system_prompt=None, **kwargs):
    """Fire the same prompt at all 3 models in parallel with streaming, display side-by-side cards."""
    kwargs.setdefault("max_tokens", 1024)
    kwargs.setdefault("extra_body", {"chat_template_kwargs": {"enable_thinking": False}})

    order = {"2B": 0, "35B": 1, "122B": 2}
    models_order = sorted(MODELS, key=lambda m: order.get(m["label"], 99))

    # Shared state for each model
    state = {}
    for m in MODELS:
        state[m["label"]] = {"text": "", "tokens": 0, "elapsed": 0, "tps": 0, "ttft": None, "done": False, "color": m["color"]}

    handle = display(HTML(_render_cards(state, models_order)), display_id=True)

    def stream_model(m):
        messages = []
        if system_prompt:
            messages.append({"role": "system", "content": system_prompt})
        messages.append({"role": "user", "content": prompt})
        t0 = time.time()
        try:
            response = clients[m["label"]].chat.completions.create(
                model=m["model"], messages=messages, stream=True, **kwargs
            )
            token_count = 0
            for chunk in response:
                if chunk.choices and chunk.choices[0].delta.content:
                    if state[m["label"]]["ttft"] is None:
                        state[m["label"]]["ttft"] = time.time() - t0
                    state[m["label"]]["text"] += chunk.choices[0].delta.content
                    token_count += 1
                    elapsed = time.time() - t0
                    state[m["label"]]["tokens"] = token_count
                    state[m["label"]]["elapsed"] = elapsed
                    state[m["label"]]["tps"] = token_count / elapsed if elapsed > 0 else 0
        except Exception as e:
            state[m["label"]]["text"] = f"Error: {e}"
        finally:
            elapsed = time.time() - t0
            state[m["label"]]["elapsed"] = elapsed
            if state[m["label"]]["tokens"] > 0:
                state[m["label"]]["tps"] = state[m["label"]]["tokens"] / elapsed
            state[m["label"]]["done"] = True

    # Launch all 3 streaming threads
    threads = []
    for m in MODELS:
        t = threading.Thread(target=stream_model, args=(m,))
        t.start()
        threads.append(t)

    # Refresh display every 300ms until all done
    while not all(state[m["label"]]["done"] for m in MODELS):
        time.sleep(0.3)
        handle.update(HTML(_render_cards(state, models_order)))

    # Final render
    handle.update(HTML(_render_cards(state, models_order)))

    # Return results
    results = []
    for m in models_order:
        s = state[m["label"]]
        results.append({
            "label": m["label"], "text": strip_think(s["text"]),
            "tokens": s["tokens"], "elapsed": s["elapsed"],
            "tps": s["tps"], "ttft": s["ttft"], "color": s["color"]
        })
    return results


def show_metrics_table(results):
    """Render a comparison metrics table from compare_models results."""
    rows = ""
    for r in results:
        ttft_ms = int(r["ttft"] * 1000) if r.get("ttft") is not None else None
        ttft_str = f"{ttft_ms} ms" if ttft_ms is not None else "—"
        rows += (
            f"<tr>"
            f"<td style='padding:6px 12px; color:{r['color']}; font-weight:bold; border-bottom:1px solid #e5e7eb;'>{r['label']}</td>"
            f"<td style='padding:6px 12px; color:#111827; border-bottom:1px solid #e5e7eb;'>{ttft_str}</td>"
            f"<td style='padding:6px 12px; color:#111827; border-bottom:1px solid #e5e7eb;'>{r['tokens']}</td>"
            f"<td style='padding:6px 12px; color:#111827; border-bottom:1px solid #e5e7eb;'>{r['elapsed']:.1f}s</td>"
            f"<td style='padding:6px 12px; color:#16a34a; font-weight:bold; border-bottom:1px solid #e5e7eb;'>{r['tps']:.1f}</td>"
            f"</tr>"
        )
    display(HTML(f"""
    <table style="background:#f9fafb; border:1px solid #d1d5db; border-collapse:collapse; border-radius:8px; overflow:hidden; margin:10px 0; font-family:monospace; min-width:400px;">
      <thead><tr style="background:#1e3a5f;">
        <th style="padding:8px 12px; color:white; text-align:left;">Model</th>
        <th style="padding:8px 12px; color:white; text-align:left;">TTFT</th>
        <th style="padding:8px 12px; color:white; text-align:left;">Tokens</th>
        <th style="padding:8px 12px; color:white; text-align:left;">Time</th>
        <th style="padding:8px 12px; color:white; text-align:left;">tok/s</th>
      </tr></thead>
      <tbody>{rows}</tbody>
    </table>
    """))

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #4f8ef7; background:#f0f4ff; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">🚀 Step 2: First Inference — 3-Model Comparison</h2>
</div>

In [84]:
results = compare_models(
    "Explain what you are in exactly 3 sentences, as if you were describing yourself to an electrical engineer who has never heard of an LLM.",
)

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #22c55e; background:#f0fdf4; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">📊 Step 3: Measure Performance</h2>
</div>

In [85]:
# Measure tok/s across all 3 models with a fun prompt
import psutil

results = compare_models(
    "Write a limerick about UART protocol timing violations.",
)

show_metrics_table(results)

ram = psutil.virtual_memory()
print(f"\nSystem RAM used: {(ram.total - ram.available) / 1e9:.1f} GB / {ram.total / 1e9:.0f} GB ({ram.percent}%)")

Model,TTFT,Tokens,Time,tok/s
2B,212 ms,1024,4.3s,235.4
122B,1252 ms,44,2.3s,18.8



System RAM used: 89.3 GB / 137 GB (65.0%)


<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #a855f7; background:#faf5ff; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">🧩 Step 4: Memory Topology</h2>
</div>

Why can Apple Silicon run all 3 models simultaneously? **Unified memory.** The CPU, GPU, and Neural Engine share the same RAM — no PCIe bottleneck, no separate VRAM pool.

In [86]:
# Show the memory topology — why Apple Silicon is special for this
import psutil
from IPython.display import HTML, display

ram = psutil.virtual_memory()
used_gb = (ram.total - ram.available) / 1e9

display(HTML(f"""
<table style="background:#f9fafb; border:1px solid #d1d5db; border-collapse:collapse; border-radius:8px; overflow:hidden; margin:10px 0; font-family:monospace; min-width:400px;">
  <thead><tr style="background:#1e3a5f;">
    <th style="padding:8px 12px; color:white; text-align:left;">Architecture</th>
    <th style="padding:8px 12px; color:white; text-align:left;">GPU Memory</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Access</th>
  </tr></thead>
  <tbody>
    <tr><td style="padding:6px 12px; color:#374151; border-bottom:1px solid #e5e7eb;">Discrete GPU</td>
        <td style="padding:6px 12px; color:#111827; font-weight:bold; border-bottom:1px solid #e5e7eb;">24 GB VRAM</td>
        <td style="padding:6px 12px; color:#dc2626; font-weight:bold; border-bottom:1px solid #e5e7eb;">Model must fit here</td></tr>
    <tr><td style="padding:6px 12px; color:#374151; border-bottom:1px solid #e5e7eb;">This Mac</td>
        <td style="padding:6px 12px; color:#111827; font-weight:bold; border-bottom:1px solid #e5e7eb;">{ram.total/1e9:.0f} GB Unified</td>
        <td style="padding:6px 12px; color:#16a34a; font-weight:bold; border-bottom:1px solid #e5e7eb;">CPU + GPU + Neural Engine share</td></tr>
  </tbody>
</table>

<table style="background:#f9fafb; border:1px solid #d1d5db; border-collapse:collapse; border-radius:8px; overflow:hidden; margin:10px 0; font-family:monospace; min-width:400px;">
  <thead><tr style="background:#1e3a5f;">
    <th style="padding:8px 12px; color:white; text-align:left;">Model</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Approx Footprint</th>
  </tr></thead>
  <tbody>
    <tr><td style="padding:6px 12px; color:#2563eb; font-weight:bold; border-bottom:1px solid #e5e7eb;">122B</td>
        <td style="padding:6px 12px; color:#111827; font-weight:bold; border-bottom:1px solid #e5e7eb;">~65 GB</td></tr>
    <tr><td style="padding:6px 12px; color:#16a34a; font-weight:bold; border-bottom:1px solid #e5e7eb;">35B</td>
        <td style="padding:6px 12px; color:#111827; font-weight:bold; border-bottom:1px solid #e5e7eb;">~20 GB</td></tr>
    <tr><td style="padding:6px 12px; color:#f59e0b; font-weight:bold; border-bottom:1px solid #e5e7eb;">2B</td>
        <td style="padding:6px 12px; color:#111827; font-weight:bold; border-bottom:1px solid #e5e7eb;">~1.2 GB</td></tr>
    <tr style="background:#f0f4ff;"><td style="padding:6px 12px; color:#1e3a5f; font-weight:bold;">Total</td>
        <td style="padding:6px 12px; color:#1e3a5f; font-weight:bold;">~86.2 GB</td></tr>
  </tbody>
</table>
"""))

print(f"System RAM in use: {used_gb:.1f} GB / {ram.total/1e9:.0f} GB ({ram.percent}%)")
print(f"Available RAM: {ram.available/1e9:.1f} GB")

System RAM in use: 88.2 GB / 137 GB (64.1%)
Available RAM: 49.3 GB


<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #4f8ef7; background:#f0f4ff; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">🌡️ Step 5: Temperature Experiment</h2>
</div>

<div class="alert alert-block alert-warning">
  <b>⚠️ Key Concept:</b> Temperature controls the randomness of token sampling. At <code>0</code> the model always picks the most likely next token (deterministic). Higher values spread probability across more tokens, making output more creative — or more chaotic.
</div>

In [87]:
# Temperature = 0.7 across all 3 models — watch the quality spread
results = compare_models(
    "In one sentence, describe how electricity flows through a wire, but make it sound like ancient mythology.",
    temperature=0,
)

# Try other temperatures yourself — change the value above and re-run!

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #a855f7; background:#faf5ff; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">🎭 Step 6: System Prompts</h2>
</div>

<div class="alert alert-block alert-warning">
  <b>⚠️ Key Concept:</b> A system prompt sets the model’s persona and behavior rules. The weights don’t change — you’re just framing the input differently. Think of it as a costume for the model.
</div>

In [89]:
# Pirate persona across all 3 models — entertaining quality spread!
results = compare_models(
    "What is a neural network?",
    system_prompt="You are a pirate who somehow understands machine learning. Arr.",
    temperature=0.1,
)

In [90]:
# Bonus: 3 personas on the 122B model — streaming in parallel
# ↓ Change this and re-run to see how temperature affects each persona ↓
PERSONA_TEMP = 0.7

import time
import threading
from IPython.display import HTML, display

personas = [
    {"name": "Grumpy Firmware Engineer", "icon": "\U0001f610", "system": "You are a grumpy firmware engineer with 30 years of experience. You think everything modern is bloated and wrong. Be brief and sarcastic.", "color": "#dc2626"},
    {"name": "Excited Intern", "icon": "\U0001f929", "system": "You are an extremely enthusiastic first-year CS student who just discovered transformers. Use lots of exclamation points.", "color": "#2563eb"},
    {"name": "ML Pirate", "icon": "\U0001f3f4\u200d\u2620\ufe0f", "system": "You are a pirate who somehow understands machine learning. Arr.", "color": "#16a34a"},
]

question = "What is a neural network?"
model_122b = next(m for m in MODELS if m["label"] == "122B")

# Shared state
state = {}
for p in personas:
    state[p["name"]] = {"text": "", "tokens": 0, "elapsed": 0, "tps": 0, "done": False}

def _render_persona_cards():
    cards = ""
    for p in personas:
        s = state[p["name"]]
        text = strip_think(s["text"])
        if not text and not s["done"]:
            rendered = "<em>waiting...</em>"
        elif not text and s["done"]:
            rendered = "<em>(empty response)</em>"
        else:
            rendered = _md(text)
        if s["done"]:
            status = f"{s['tps']:.1f} tok/s"
        else:
            status = f"streaming... {s['tokens']} tok"
        cards += f"""
        <div style="flex:1; min-width:250px; background:#f9fafb; border:1px solid #d1d5db; border-radius:8px;
                    padding:16px 20px; font-family:'Inter',sans-serif; word-wrap:break-word; overflow-wrap:break-word;">
          <div style="color:{p['color']}; font-weight:bold; font-size:0.9em; margin-bottom:8px;">
            {p['icon']} {p['name']} · {status}
          </div>
          <div style="color:#1f2937; line-height:1.6;">{rendered}</div>
        </div>"""
    return f"""<div style="color:#6b7280; font-size:0.85em; margin-bottom:8px;">122B model — same weights, different personas (temp={PERSONA_TEMP}):</div>
<div style="display:flex; gap:16px; flex-wrap:wrap; margin:10px 0;">{cards}</div>"""

handle = display(HTML(_render_persona_cards()), display_id=True)

def stream_persona(p):
    t0 = time.time()
    try:
        response = clients["122B"].chat.completions.create(
            model=model_122b["model"],
            messages=[
                {"role": "system", "content": p["system"]},
                {"role": "user", "content": question},
            ],
            max_tokens=512, temperature=PERSONA_TEMP, stream=True,
            extra_body={"chat_template_kwargs": {"enable_thinking": False}},
        )
        token_count = 0
        for chunk in response:
            if chunk.choices and chunk.choices[0].delta.content:
                state[p["name"]]["text"] += chunk.choices[0].delta.content
                token_count += 1
                elapsed = time.time() - t0
                state[p["name"]]["tokens"] = token_count
                state[p["name"]]["elapsed"] = elapsed
                state[p["name"]]["tps"] = token_count / elapsed if elapsed > 0 else 0
    except Exception as e:
        state[p["name"]]["text"] = f"Error: {e}"
    finally:
        elapsed = time.time() - t0
        state[p["name"]]["elapsed"] = elapsed
        if state[p["name"]]["tokens"] > 0:
            state[p["name"]]["tps"] = state[p["name"]]["tokens"] / elapsed
        state[p["name"]]["done"] = True

threads = [threading.Thread(target=stream_persona, args=(p,)) for p in personas]
for t in threads:
    t.start()

while not all(state[p["name"]]["done"] for p in personas):
    time.sleep(0.3)
    handle.update(HTML(_render_persona_cards()))

handle.update(HTML(_render_persona_cards()))

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #a855f7; background:#faf5ff; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">🧮 Step 7: Quantization</h2>
</div>

<div class="alert alert-block alert-warning">
  <b>⚠️ Key Concept:</b> Quantization reduces the precision of model weights (e.g., from 32-bit floats to 4-bit integers). This shrinks the model ~8× with only a small quality loss, making it possible to fit large models in memory.
</div>

In [91]:
# What quantization actually does to numbers
import mlx.core as mx
from IPython.display import HTML, display

# Real weights are fp32 — 4 bytes each
weights_fp32 = mx.array([0.1234, -0.5678, 0.9012, -0.3456, 0.7890])

# 4-bit quantization: ~16 possible values per weight, loses precision
# Simulate by rounding to nearest 1/8
weights_4bit_approx = mx.round(weights_fp32 * 8) / 8

print("What quantization does to weights:")
print(f"  fp32:   {weights_fp32.tolist()}")
print(f"  ~4-bit: {weights_4bit_approx.tolist()}")
print()

# Size math — this is why quantization matters
params = 32_000_000_000  # 32B parameters

rows = ""
for label, mult, note in [
    ("fp32 (4 bytes/param)", 4, "doesn't fit on most machines"),
    ("bf16 (2 bytes/param)", 2, "still large"),
    ("4-bit (0.5 bytes/param)", 0.5, "fits in 64GB+ Mac"),
]:
    size = params * mult / 1e9
    rows += (
        f"<tr><td style='padding:6px 12px; color:#374151; border-bottom:1px solid #e5e7eb;'>{label}</td>"
        f"<td style='padding:6px 12px; color:#111827; font-weight:bold; border-bottom:1px solid #e5e7eb;'>{size:.0f} GB \u2014 {note}</td></tr>"
    )

display(HTML(f"""
<table style="background:#f9fafb; border:1px solid #d1d5db; border-collapse:collapse; border-radius:8px; overflow:hidden; margin:10px 0; font-family:monospace; min-width:400px;">
  <thead><tr style="background:#1e3a5f;">
    <th style="padding:8px 12px; color:white; text-align:left;">Precision</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Size (32B params)</th>
  </tr></thead>
  <tbody>{rows}</tbody>
</table>
"""))

What quantization does to weights:
  fp32:   [0.1234000027179718, -0.567799985408783, 0.901199996471405, -0.3456000089645386, 0.7889999747276306]
  ~4-bit: [0.125, -0.625, 0.875, -0.375, 0.75]



Precision,Size (32B params)
fp32 (4 bytes/param),128 GB — doesn't fit on most machines
bf16 (2 bytes/param),64 GB — still large
4-bit (0.5 bytes/param),16 GB — fits in 64GB+ Mac


<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #22c55e; background:#f0fdf4; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">💪 Step 8: Stress Test</h2>
</div>

In [ ]:
# Stress test: 3 tests x N models — progressive results with spinners
import time, threading
import psutil
from IPython.display import HTML, display, update_display

tests = [
    ("Simple", "What is 2+2? Answer with just the number.", 64),
    ("Medium", "List 5 common I2C bus errors and how to debug them.", 512),
    ("Creative", "Write a Python function that reads from a UART buffer and parses NMEA GPS sentences. Include docstring.", 1024),
]

_CSS = "<style>@keyframes spin{to{transform:rotate(360deg)}}.sp{display:inline-block;width:14px;height:14px;border:2px solid #ccc;border-top-color:#555;border-radius:50%;animation:spin .8s linear infinite;vertical-align:middle}</style>"

order = {"2B": 0, "35B": 1, "122B": 2}
models_sorted = sorted(MODELS, key=lambda m: order.get(m["label"], 99))

# State: test_name -> label -> {status, tokens, elapsed, tps}
stress_state = {}
for test_name, _, _ in tests:
    stress_state[test_name] = {}
    for m in MODELS:
        stress_state[test_name][m["label"]] = {"status": "pending", "tokens": 0, "elapsed": 0, "tps": 0}

current_test_idx = 0

def _render_stress():
    html = _CSS
    for i, (test_name, _, _) in enumerate(tests):
        rows = []
        for m in models_sorted:
            label = m["label"]
            color = m["color"]
            st = stress_state[test_name][label]
            if st["status"] == "pending":
                tok_cell = "—"
                time_cell = "—"
                tps_cell = "—"
            elif st["status"] == "running":
                tok_cell = '<span class="sp"></span>'
                time_cell = '<span class="sp"></span>'
                tps_cell = '<span class="sp"></span>'
            elif st["status"] == "done":
                tok_cell = str(st["tokens"])
                time_cell = f"{st['elapsed']:.1f}s"
                tps_cell = f"<span style='color:#16a34a;font-weight:bold'>{st['tps']:.1f}</span>"
            else:
                tok_cell = "err"
                time_cell = "—"
                tps_cell = "—"
            rows.append(
                f'<tr>'
                f'<td style="padding:6px 12px;font-weight:bold;color:{color}">{label}</td>'
                f'<td style="padding:6px 12px">{tok_cell}</td>'
                f'<td style="padding:6px 12px">{time_cell}</td>'
                f'<td style="padding:6px 12px">{tps_cell}</td>'
                f'</tr>'
            )
        body = "\n".join(rows)
        html += (
            f'<div style="color:#1e3a5f;font-weight:bold;margin:8px 0 4px 0">{test_name}</div>'
            + '<table style="border-collapse:collapse;font-family:monospace;font-size:14px;'
            + 'background:#f9fafb;border:1px solid #d1d5db;border-radius:8px;overflow:hidden;margin:4px 0 12px 0;min-width:400px">'
            + '<thead><tr style="background:#1e3a5f">'
            + '<th style="padding:8px 12px;color:white;text-align:left">Model</th>'
            + '<th style="padding:8px 12px;color:white;text-align:left">Tokens</th>'
            + '<th style="padding:8px 12px;color:white;text-align:left">Time</th>'
            + '<th style="padding:8px 12px;color:white;text-align:left">tok/s</th>'
            + '</tr></thead>'
            + '<tbody>' + body + '</tbody></table>'
        )
    return html

def run_test(m, test_name, prompt, max_tokens):
    label = m["label"]
    stress_state[test_name][label]["status"] = "running"
    try:
        t0 = time.time()
        r = clients[label].chat.completions.create(
            model=m["model"],
            messages=[{"role": "user", "content": prompt}],
            max_tokens=max_tokens,
            extra_body={"chat_template_kwargs": {"enable_thinking": False}},
        )
        elapsed = time.time() - t0
        tokens = r.usage.completion_tokens
        stress_state[test_name][label]["tokens"] = tokens
        stress_state[test_name][label]["elapsed"] = elapsed
        stress_state[test_name][label]["tps"] = tokens / elapsed if elapsed > 0 else 0
        stress_state[test_name][label]["status"] = "done"
    except Exception:
        stress_state[test_name][label]["status"] = "failed"

dh = display(HTML(_render_stress()), display_id=True)

for test_name, prompt, max_tok in tests:
    threads = []
    for m in MODELS:
        t = threading.Thread(target=run_test, args=(m, test_name, prompt, max_tok), daemon=True)
        t.start()
        threads.append(t)

    while any(t.is_alive() for t in threads):
        time.sleep(0.5)
        update_display(HTML(_render_stress()), display_id=dh.display_id)

    for t in threads:
        t.join()
    update_display(HTML(_render_stress()), display_id=dh.display_id)

update_display(HTML(_render_stress()), display_id=dh.display_id)

ram = psutil.virtual_memory()
print(f"All stress tests done.")
print(f"System RAM used: {(ram.total - ram.available) / 1e9:.1f} GB / {ram.total / 1e9:.0f} GB ({ram.percent}%)")


Running: Simple (max_tokens=64)...


Model,Tokens,Time,tok/s
2B,2,0.2s,9.2
122B,4,1.2s,3.3


  ✓ Simple complete

Running: Medium (max_tokens=512)...


<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #a855f7; background:#faf5ff; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">🗂️ Step 9: KV Cache — Why Second Turns Are Faster</h2>
</div>

<div class="alert alert-block alert-warning">
  <b>⚠️ Key Concept:</b> The KV (key-value) cache stores attention matrices from prior tokens so the model doesn’t recompute them on every turn. This speeds up multi-turn conversations but grows with context length — long conversations eat more RAM.
</div>

In [ ]:
# KV cache demo: measure time-to-first-token (TTFT) for cold vs cached
# TTFT isolates prompt processing time — the phase KV cache speeds up
import time, threading
from IPython.display import HTML, display, update_display

_CSS = "<style>@keyframes spin{to{transform:rotate(360deg)}}.sp{display:inline-block;width:14px;height:14px;border:2px solid #ccc;border-top-color:#555;border-radius:50%;animation:spin .8s linear infinite;vertical-align:middle}</style>"

kv_state = {}
for m in MODELS:
    kv_state[m["label"]] = {"status": "testing", "cold": 0, "cached": 0, "speedup": 0, "start": time.time()}

def _render_kv():
    rows = []
    for m in MODELS:
        label = m["label"]
        color = m["color"]
        st = kv_state[label]
        elapsed = time.time() - st["start"]
        if st["status"] == "testing":
            cold_cell = '<span class="sp"></span>'
            cached_cell = "—"
            speedup_cell = "—"
            el = f"{elapsed:.0f}s"
        elif st["status"] == "done":
            cold_cell = f"{st['cold']*1000:.0f} ms"
            cached_cell = f"{st['cached']*1000:.0f} ms"
            speedup_cell = f"<span style='color:#16a34a;font-weight:bold'>{st['speedup']:.1f}x</span>"
            el = ""
        else:
            cold_cell = "failed"
            cached_cell = "—"
            speedup_cell = "—"
            el = ""
        rows.append(
            f'<tr>'
            f'<td style="padding:6px 12px;font-weight:bold;color:{color}">{label}</td>'
            f'<td style="padding:6px 12px">{cold_cell}</td>'
            f'<td style="padding:6px 12px">{cached_cell}</td>'
            f'<td style="padding:6px 12px">{speedup_cell}</td>'
            f'<td style="padding:6px 12px;color:#888">{el}</td>'
            f'</tr>'
        )
    body = "\n".join(rows)
    return (
        _CSS
        + '<table style="border-collapse:collapse;font-family:monospace;font-size:14px;'
        + 'background:#f9fafb;border:1px solid #d1d5db;border-radius:8px;overflow:hidden;min-width:500px">'
        + '<thead><tr style="background:#1e3a5f">'
        + '<th style="padding:8px 12px;color:white;text-align:left">Model</th>'
        + '<th style="padding:8px 12px;color:white;text-align:left">TTFT cold</th>'
        + '<th style="padding:8px 12px;color:white;text-align:left">TTFT cached</th>'
        + '<th style="padding:8px 12px;color:white;text-align:left">Speedup</th>'
        + '<th style="padding:8px 12px;color:white;text-align:left"></th>'
        + '</tr></thead>'
        + '<tbody>' + body + '</tbody></table>'
        + '<div style="color:#6b7280;font-size:0.8em;margin-top:4px">'
        + 'TTFT = time-to-first-token. Cached turn reuses KV matrices from turn 1.</div>'
    )

def kv_cache_test(m):
    label = m["label"]
    kv_state[label]["start"] = time.time()
    try:
        # Turn 1 — cold
        messages_1 = [{"role": "user", "content": "I'm going to ask you 3 questions about embedded systems. Ready?"}]
        t0 = time.time()
        stream_1 = clients[label].chat.completions.create(
            model=m["model"], messages=messages_1, max_tokens=50, stream=True,
            extra_body={"chat_template_kwargs": {"enable_thinking": False}},
        )
        ttft_cold = None
        reply_1 = ""
        for chunk in stream_1:
            if ttft_cold is None and chunk.choices and chunk.choices[0].delta.content:
                ttft_cold = time.time() - t0
            if chunk.choices and chunk.choices[0].delta.content:
                reply_1 += chunk.choices[0].delta.content
        if ttft_cold is None:
            ttft_cold = time.time() - t0

        # Turn 2 — cached
        messages_2 = messages_1 + [
            {"role": "assistant", "content": reply_1},
            {"role": "user", "content": "Question 1: What's the difference between I2C and SPI?"},
        ]
        t1 = time.time()
        stream_2 = clients[label].chat.completions.create(
            model=m["model"], messages=messages_2, max_tokens=50, stream=True,
            extra_body={"chat_template_kwargs": {"enable_thinking": False}},
        )
        ttft_cached = None
        for chunk in stream_2:
            if ttft_cached is None and chunk.choices and chunk.choices[0].delta.content:
                ttft_cached = time.time() - t1
                break
        if ttft_cached is None:
            ttft_cached = time.time() - t1

        speedup = ttft_cold / ttft_cached if ttft_cached > 0 else 0
        kv_state[label]["cold"] = ttft_cold
        kv_state[label]["cached"] = ttft_cached
        kv_state[label]["speedup"] = speedup
        kv_state[label]["status"] = "done"
    except Exception:
        kv_state[label]["status"] = "failed"

dh = display(HTML(_render_kv()), display_id=True)

threads = [threading.Thread(target=kv_cache_test, args=(m,), daemon=True) for m in MODELS]
for t in threads:
    t.start()

while any(t.is_alive() for t in threads):
    time.sleep(0.5)
    update_display(HTML(_render_kv()), display_id=dh.display_id)

for t in threads:
    t.join()

update_display(HTML(_render_kv()), display_id=dh.display_id)
print("Done.")


<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #22c55e; background:#f0fdf4; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">📝 Takeaways</h2>
</div>

- **Unified memory** lets Apple Silicon run models that don’t fit on discrete GPUs
- **MLX** serves models locally via an OpenAI-compatible API
- **4-bit quantization** shrinks models 8× with minimal quality loss
- **Temperature** controls randomness: 0 = deterministic, 1.5 = chaotic
- **System prompts** steer behavior without changing model weights
- **tok/s** is bounded by memory bandwidth, not compute
- **KV cache** speeds up multi-turn conversations by reusing prior computations
- **Running multiple models simultaneously** reveals the quality/speed tradeoff viscerally
- **Model size vs quality:** 0.8B is instant but shallow, 35B hits the sweet spot, 122B is best but slowest
- Everything here ran locally — no data left your machine

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #4f8ef7; background:#f0f4ff; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">🚀 What’s Next</h2>
</div>

- **Section 01b:** Model Datasheet — architecture deep dive into what we're actually running (DeltaNet/GQA hybrid, MoE, KV cache math)
- **Section 01c:** Inference Optimization — speculative decoding, prefix caching, quantization formats, continuous batching
- **Section 02:** Structured output — making LLMs return JSON you can actually parse
- **Section 03:** Tool use — letting models call functions and interact with external systems